In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv(r'E:\Air-Quality-Forecasting\data\raw_data\pollution_data\pollution_us_2000_2016.csv')

In [4]:
df.head()

,Unnamed: 0,State Code,County Code,Site Num,Address,State,County,City,Date Local,NO2 Units,...,SO2 Units,SO2 Mean,SO2 1st Max Value,SO2 1st Max Hour,SO2 AQI,CO Units,CO Mean,CO 1st Max Value,CO 1st Max Hour,CO AQI
0,0,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-01,Parts per billion,...,Parts per billion,3.000000,9.0,21,13.0,Parts per million,1.145833,4.2,21,NaN
1,1,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-01,Parts per billion,...,Parts per billion,3.000000,9.0,21,13.0,Parts per million,0.878947,2.2,23,25.0
2,2,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-01,Parts per billion,...,Parts per billion,2.975000,6.6,23,NaN,Parts per million,1.145833,4.2,21,NaN
3,3,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-01,Parts per billion,...,Parts per billion,2.975000,6.6,23,NaN,Parts per million,0.878947,2.2,23,25.0
4,4,4,13,3002,1645 E ROOSEVELT ST-CENTRAL PHOENIX STN,Arizona,Maricopa,Phoenix,2000-01-02,Parts per billion,...,Parts per billion,1.958333,3.0,22,4.0,Parts per million,0.850000,1.6,23,NaN


In [5]:
df.shape

(1746661, 29)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1746661 entries, 0 to 1746660
Data columns (total 29 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Unnamed: 0         int64  
 1   State Code         int64  
 2   County Code        int64  
 3   Site Num           int64  
 4   Address            object 
 5   State              object 
 6   County             object 
 7   City               object 
 8   Date Local         object 
 9   NO2 Units          object 
 10  NO2 Mean           float64
 11  NO2 1st Max Value  float64
 12  NO2 1st Max Hour   int64  
 13  NO2 AQI            int64  
 14  O3 Units           object 
 15  O3 Mean            float64
 16  O3 1st Max Value   float64
 17  O3 1st Max Hour    int64  
 18  O3 AQI             int64  
 19  SO2 Units          object 
 20  SO2 Mean           float64
 21  SO2 1st Max Value  float64
 22  SO2 1st Max Hour   int64  
 23  SO2 AQI            float64
 24  CO Units           object 
 25  CO Mean           

In [8]:
df["Date Local"] = pd.to_datetime(df["Date Local"])

df["Year"] = df["Date Local"].dt.year

print(df.groupby("Year").size())

Year
2000     91646
2001     89471
2002     93542
2003     86398
2004     82164
2005     85245
2006    100839
2007    115378
2008    119549
2009    104175
2010    103975
2011    129248
2012    128206
2013    134576
2014    131589
2015    126056
2016     24604
dtype: int64


In [9]:
df['station'] = (df['State Code'].astype(str) + '_' +
                 df['County Code'].astype(str) + '_' +
                 df['Site Num'].astype(str))

In [10]:
df = df[(df['Date Local'] >= '2010-01-01') & (df['Date Local'] <= '2015-12-31')]

In [11]:
df = df.groupby(['station','State','City','Date Local']).mean(numeric_only=True).reset_index()

In [12]:
counts = df.groupby('station').size()
good_stations = counts[counts >= 1500].index
df = df[df['station'].isin(good_stations)]

In [13]:
station_state = df.drop_duplicates('station')[['station','State']]
sampled = (station_state.groupby('State', group_keys=False)
           .apply(lambda x: x.sample(min(len(x), 2), random_state=42)))

E:\temp\ipykernel_12824\952822715.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 2), random_state=42)))


In [14]:
df_sample = df[df['station'].isin(sampled['station'])]
print(len(df_sample), df_sample['station'].nunique(), df_sample['State'].nunique())

70922 37 29


In [17]:
df_sample = df_sample.sort_values(['station', 'Date Local']).reset_index(drop=True)
df_sample.to_csv(r"E:\Air-Quality-Forecasting\data\raw_data\pollution_sample.csv", index=False)